# Phase 3: the last GPU run (Kaggle, resumable)

One session, three parts, each scored by the grounder (the dev-chosen adapter) and pushed to the Hub as it goes:

| part | what | why |
|---|---|---|
| `guard_dev` | 300 sandbox **dev** items (pages 100-159) | the grounder's answers a better guard is designed on -- never test pages |
| `realtest` | Test C: real apps, boxes drawn by a person | genuinely UIA-blind targets, truth from neither method |
| ScreenSpot-v2 | 1,272 public items, corrected labels | the public number on the cleaned benchmark |

Everything about the tree (name matching, the LLM, the guards) is scored afterwards on CPU from these answers.

## Setup
Secret `HF_TOKEN` attached, **GPU T4 x2**, Internet **On**, **Input: the `compuse-final` dataset only**.

## Run: Save Version -> "Save & Run All (Commit)" (~3 h). It stops itself if the grounder's first 20 answers don't parse.
If the session dies, run it again: finished answers are on the Hub and are skipped.

In [ ]:
# Cell 1 -- settings. Runs before ANY torch/transformers import, on purpose.
import os
import time

SESSION_START = time.time()

RUN_MODE = "eval"                 # "eval" (everything) or "smoke" (a few items per part)
PARTS = ("guard_dev", "realtest", "screenspot_v2")   # drop one to skip it
HF_REPO_NAME = "computer-use-grounder"   # the adapter comes from it, results go back to it
CODE_REF = "c089d74f6e0060bdda35be183691405c4073bfc2"           # exact commit of rudranaresh0201/computer-use

assert RUN_MODE in ("smoke", "eval"), RUN_MODE
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"RUN_MODE={RUN_MODE}  PARTS={PARTS}  CODE_REF={CODE_REF}")

In [ ]:
# Cell 2 -- the GPU the model was trained and evaluated on.
import subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0, "No GPU attached. Session options -> Accelerator -> GPU T4 x2."
assert "T4" in smi.stdout, "Expected a Tesla T4 (Session options -> Accelerator -> GPU T4 x2)."

In [ ]:
# Cell 3 -- the token must work in THIS committed session and be able to write results.
import requests
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Could not read the HF_TOKEN secret. Add-ons -> Secrets: add HF_TOKEN, tick it for this notebook.") from e
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN secret is empty or not a Hugging Face token"
r = requests.get("https://huggingface.co/api/whoami-v2", headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
assert r.status_code == 200, f"Hugging Face rejected the token (HTTP {r.status_code}): {r.text[:200]}"
me = r.json()
assert me.get("auth", {}).get("accessToken", {}).get("role") != "read", "HF_TOKEN is READ-only; results need write."
HF_REPO = f"{me['name']}/{HF_REPO_NAME}"
PREFIX = "final" if RUN_MODE == "eval" else f"final-smoke-{CODE_REF[:7]}"
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"token OK for {me['name']}; adapter from {HF_REPO}, results to {HF_REPO}/{PREFIX}/")

In [ ]:
# Cell 4 -- the exact code version, and its pinned dependencies.
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/computer-use")
if not REPO_DIR.is_dir():
    !git clone https://github.com/rudranaresh0201/computer-use.git {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin
!git -C {REPO_DIR} checkout --quiet {CODE_REF}
!git -C {REPO_DIR} log -1 --format="evaluating with commit %H (%s)"
!{sys.executable} -m pip install --quiet -e "{REPO_DIR}[training]"
!{sys.executable} -m pip uninstall -y --quiet torchao

In [ ]:
# Cell 5 -- imports at the pinned versions, and the adapter our own dev split chose.
import importlib
import json

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()

import torch
import transformers

from computeruse.training.hub_checkpoint import HubSync

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
assert transformers.__version__ == "5.0.0", "pinned transformers did not load -- restart and run from the top"
assert torch.cuda.is_available()

sync = HubSync(HF_REPO, token=HF_TOKEN)
print(f"Hub write check OK as {sync.preflight()}")
STORE_DIR = Path("/kaggle/working/final_store")
assert sync.fetch(["eval/selection.json"], STORE_DIR), "no eval/selection.json -- run the v3 eval notebook first"
CHOSEN = json.loads((STORE_DIR / "eval" / "selection.json").read_text())["chosen"]
print(f"adapter chosen on our dev split: {CHOSEN}")

In [ ]:
# Cell 6 -- the input parts. Each folder with a bundle.json is one part.
BUNDLES = {}
for marker in sorted(Path("/kaggle/input").rglob("bundle.json")):
    meta = json.loads(marker.read_text())
    items = [json.loads(line) for line in (marker.parent / "items.jsonl").read_text().splitlines() if line.strip()]
    assert len(items) == meta["items"], f"{marker}: items.jsonl does not match bundle.json"
    BUNDLES[meta["name"]] = (marker.parent, items)
for name in ("guard_dev", "realtest"):
    if name not in PARTS:
        BUNDLES.pop(name, None)
    elif name not in BUNDLES:
        print(f"WARNING: part {name!r} is not in the input dataset -- it will be skipped")
if RUN_MODE == "smoke":
    BUNDLES = {n: (root, [i for c in sorted({x["condition"] for x in items})
                          for i in [x for x in items if x["condition"] == c][:3]]) for n, (root, items) in BUNDLES.items()}
for name, (root, items) in BUNDLES.items():
    counts = {}
    for i in items:
        counts[i["condition"]] = counts.get(i["condition"], 0) + 1
    print(f"{name}: {len(items)} items {dict(sorted(counts.items()))}  ({root})")
assert BUNDLES or "screenspot_v2" in PARTS, "nothing to score"

In [ ]:
# Cell 7 -- base model + the one chosen adapter.
from transformers import AutoModelForImageTextToText, AutoProcessor

from computeruse.eval.ablation import adapter_path
from computeruse.eval.resumable import ResultStore, score_arm
from computeruse.eval.vlm_grounder import MODEL_ID, ZERO_SHOT, arm_active, format_counts, load_adapters, summarize_diagnostics
from computeruse.training.prepare_dataset import TrainingExample
from computeruse.training.train_lora import resolve_precision, torch_dtype_for

ADAPTER_ROOT = Path("/kaggle/working/adapters")
folder = "final" if CHOSEN == "final" else f"adapters/{CHOSEN}"
assert sync.fetch([f"{folder}/adapter_*"], ADAPTER_ROOT), f"adapter {CHOSEN} not found on the Hub"

precision = resolve_precision()
processor = AutoProcessor.from_pretrained(MODEL_ID)
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch_dtype_for(precision), attn_implementation="sdpa"
).to("cuda")
model = load_adapters(base, {CHOSEN: adapter_path(CHOSEN, ADAPTER_ROOT)})
del base  # the plain base model is now only arm_active(model, ZERO_SHOT)
print(f"{MODEL_ID} in {precision} + {CHOSEN}; GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")
SANITY_DONE = False


def sanity_check(store, arm, examples, score):
    """The first 20 answers must parse, or the run stops before spending the budget."""
    global SANITY_DONE
    if SANITY_DONE:
        return
    with arm_active(model, arm), torch.no_grad():
        first = score_arm(store, arm, examples[:20], score)
    rate = summarize_diagnostics(first)["parse_rate"]
    print(f"sanity check -- {arm} parse rate on the first {len(first)}: {rate:.0%}")
    for r in first[:4]:
        print(f"  {r.split:<10} -> {r.predicted_text!r} hit={r.hit}")
    assert rate >= 0.8, "the grounder's answers are not parsing -- stopping before spending the GPU budget"
    SANITY_DONE = True

In [ ]:
# Cell 8 -- guard_dev (fine-tuned only: the guard decides between the tree and THIS grounder),
# then realtest (fine-tuned and zero-shot: does training help on genuinely UIA-blind targets?).
from computeruse.eval import compare

ARMS_OF = {"guard_dev": (CHOSEN,), "realtest": (CHOSEN, ZERO_SHOT)}
PART_RESULTS = {}
for name, (root, items) in BUNDLES.items():
    store = ResultStore(sync, f"{PREFIX}/{name}", STORE_DIR, flush_every_seconds=300)
    print(f"\n== {name}: {len(items)} items; already on the Hub: {store.pull()}")
    by_id = {i["id"]: i for i in items}
    examples = [TrainingExample(i["id"], i["image"], i["instruction"], "", i["condition"], i["condition"]) for i in items]

    def score(example, root=root, by_id=by_id):
        return compare.score_vlm(model, processor, by_id[example.id], root)

    sanity_check(store, CHOSEN, examples, score)
    PART_RESULTS[name] = {}
    for arm in ARMS_OF[name]:
        t0 = time.time()
        with arm_active(model, arm), torch.no_grad():
            results = score_arm(store, arm, examples, score)
        PART_RESULTS[name][arm] = results
        table = {}
        for r in results:
            table.setdefault(r.split, []).append(r.hit)
        row = "  ".join(f"{c} {sum(h) / len(h):.0%} (n={len(h)})" for c, h in sorted(table.items()))
        print(f"{name} / {arm}: {sum(r.hit for r in results) / len(results):.1%} overall | {row} "
              f"| {(time.time() - t0) / 60:.0f} min | formats {format_counts(results)}")
    store.flush()

In [ ]:
# Cell 9 -- ScreenSpot-v2: the same 1,272 instructions as ScreenSpot with corrected boxes (~1.3 GB of images).
from computeruse.eval import screenspot

if "screenspot_v2" in PARTS:
    SS_ITEMS = screenspot.load_v2(cache_dir=Path("/kaggle/working/hf_cache"))
    print(f"ScreenSpot-v2: {len(SS_ITEMS)} examples")
    assert len(SS_ITEMS) == screenspot.V2_EXPECTED_ROWS, f"expected {screenspot.V2_EXPECTED_ROWS} rows"
    if RUN_MODE == "smoke":
        by_slice = {}
        for item in SS_ITEMS:
            by_slice.setdefault(item.slice, []).append(item)
        SS_ITEMS = [i for group in by_slice.values() for i in group[:4]]
    ss_store = ResultStore(sync, f"{PREFIX}/screenspot_v2", STORE_DIR, flush_every_seconds=300)
    print(f"already on the Hub: {ss_store.pull()}")

    def ss_score(item):
        return screenspot.score_item(model, processor, item)

    sanity_check(ss_store, CHOSEN, SS_ITEMS, ss_score)
    SS_RESULTS = {}
    for arm in (CHOSEN, ZERO_SHOT):
        with arm_active(model, arm), torch.no_grad():
            SS_RESULTS[arm] = score_arm(ss_store, arm, SS_ITEMS, ss_score)
    SS_REPORT = screenspot.report({ZERO_SHOT: SS_RESULTS[ZERO_SHOT], "fine_tuned": SS_RESULTS[CHOSEN]})
    SS_REPORT.update({"adapter": CHOSEN, "code_ref": CODE_REF, "examples": len(SS_ITEMS), "benchmark": "ScreenSpot-v2",
                      "zero_shot_formats": format_counts(SS_RESULTS[ZERO_SHOT])})
    ss_store.push_json("report.json", SS_REPORT)

    columns = [f"{p}/{t}" for p in screenspot.PLATFORMS for t in screenspot.TYPES] + list(screenspot.PLATFORMS) + ["overall"]

    def fmt(cell):
        return "-" if cell["accuracy"] is None else f"{cell['accuracy']:.1%}"

    print(f"\nScreenSpot-v2 click accuracy (fine_tuned = {CHOSEN})")
    print(f"{'arm':<11}" + "".join(f"{c:>15}" for c in columns))
    for arm in (ZERO_SHOT, "fine_tuned"):
        print(f"{arm:<11}" + "".join(f"{fmt(SS_REPORT['arms'][arm][c]):>15}" for c in columns))
    print("\nfine-tuned minus zero-shot, paired, 95% CI:")
    for group, d in SS_REPORT["vs_zero_shot"]["fine_tuned"].items():
        lo, hi = d["ci95"]
        print(f"  {group:<8} {d['diff']:+.1%} [{lo:+.1%}, {hi:+.1%}]  n={d['n']}")

In [ ]:
# Cell 10 -- done. The tree methods and guards are scored on CPU from these answers.
print("\n" + "=" * 70)
hours = (time.time() - SESSION_START) / 3600
done = list(BUNDLES) + (["screenspot_v2"] if "screenspot_v2" in PARTS else [])
print(("FINAL SMOKE PASSED" if RUN_MODE == "smoke" else "FINAL RUN FINISHED")
      + f" in {hours:.1f} h. Parts: {done}. Results: {HF_REPO}/{PREFIX}/")
print("=" * 70)